In [2]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "PAQ_J.xpt"
csv_file = DATA_DIR / "PAQ_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2017-2018 PAQ_J XPT -> CSV")
print("=" * 90)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "PAQ605",
    "PAQ610",
    "PAD615",
    "PAQ620",
    "PAQ625",
    "PAD630",
    "PAQ635",
    "PAQ640",
    "PAD645",
    "PAQ650",
    "PAQ655",
    "PAD660",
    "PAQ665",
    "PAQ670",
    "PAD675",
    "PAD680"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "PAQ_J column names/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 5856:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 17:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 5,856 rows x 17 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached PAQ_J shows:
#
#     no genuine fractional observations
#
# All released variables are whole-number fields:
#
#     respondent ID
#     Yes/No response codes
#     number of days
#     number of minutes
#
# Therefore these variables may be converted to nullable
# Int64 AFTER proving that they are integral.
#
# IMPORTANT:
#
# This conclusion applies ONLY to PAQ_J.
#
# It does NOT apply to genuine decimal fields in:
#
#     Dietary
#     Laboratory
#     Examination
#     survey weights
#     nutrients
#     environmental measurements
#
# Those values must remain untouched.
#

DECIMAL_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas decoding artifact:
#
#     5.397605346934028e-79
#
# This commonly represents numeric zero in affected
# NHANES XPT files.
#
# NEVER blindly do:
#
#     df.replace(TINY_VALUE, 0)
#
# without first validating where the value occurs.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 17 PAQ_J variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE ATTACHED FILE TINY-VALUE PATTERN
# ============================================================

#
# Direct inspection of attached PAQ_J.xpt:
#
#     PAD680 = 3 tiny values
#
#     TOTAL = 3
#
# CDC defines PAD680 as sedentary minutes and permits:
#
#     0 to 1320 minutes
#
# Therefore zero is a legitimate PAD680 value.
#
# CDC publishes the complete 0-1320 range count rather
# than a separate count specifically for PAD680=0.
#
# We therefore validate:
#
#     1. tiny values occur ONLY in PAD680
#     2. there are exactly 3 in this attachment
#     3. zero is CDC-valid for PAD680
#     4. all CDC aggregate frequencies/ranges still match
#

EXPECTED_TINY_COUNTS = {
    "PAD680": 3
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated PAQ_J attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 3:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros "
        "already existed before correction."
    )


print(
    "\nPASS: Exact PAD680 3-value "
    "tiny-zero artifact pattern validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED PAD680 ZEROS
# ============================================================

pad680_tiny_mask = (
    df["PAD680"]
    .eq(TINY_VALUE)
)


correction_count = int(
    pad680_tiny_mask.sum()
)


if correction_count != 3:

    raise ValueError(
        "Unexpected PAD680 tiny-value count."
    )


df.loc[
    pad680_tiny_mask,
    "PAD680"
] = 0


print("\n--- VALIDATED ZERO RESTORATION ---")

print(
    "PAD680 restored to zero:",
    correction_count
)


if int(
    df["PAD680"]
    .eq(0)
    .sum()
) != 3:

    raise ValueError(
        "PAD680 zero restoration failed."
    )


print(
    "PASS: 3 PAD680 XPORT zero artifacts restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE
# ============================================================

#
# This represents exactly what the corrected XPT-derived
# dataset should contain.
#
# The ONLY permitted numeric modification:
#
#     PAD680 tiny value -> 0
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["PAD680"]
    .eq(TINY_VALUE),
    "PAD680"
] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. CHECK FOR GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# CRITICAL:
#
# This occurs BEFORE any Int64 conversion.
#
# np.round() below is ONLY used for comparison.
#
# It does NOT modify df.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) == 0:

        continue

    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )

    fractional_count = int(
        fractional_mask.sum()
    )

    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )

    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal variables unexpectedly "
        "found in PAQ_J.\n\n"
        "They will NOT be rounded or converted to Int64:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 11. VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93705:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 93705-102956 validated."
)


# ============================================================
# 12. CONVERT PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# Every PAQ_J field was checked above.
#
# NO .round() operation is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 17 PAQ_J variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 13. EXACT CDC FREQUENCY HELPER
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )

    print(
        f"\n--- CDC CHECK: {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )

    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )

    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )

    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 14. CDC YES/NO FREQUENCY CHECKS
# ============================================================

check_frequency(
    "PAQ605",
    {
        1: 1389,
        2: 4461,
        9: 6
    },
    0
)


check_frequency(
    "PAQ620",
    {
        1: 2439,
        2: 3412,
        9: 5
    },
    0
)


check_frequency(
    "PAQ635",
    {
        1: 1439,
        2: 4417
    },
    0
)


check_frequency(
    "PAQ650",
    {
        1: 1434,
        2: 4422
    },
    0
)


check_frequency(
    "PAQ665",
    {
        1: 2308,
        2: 3548
    },
    0
)


# ============================================================
# 15. CDC RANGE-SUMMARY HELPER
# ============================================================

def check_range_summary(
    column,
    lower,
    upper,
    expected_range_count,
    special_counts,
    expected_missing
):

    values = df[column]

    range_count = int(
        values.between(
            lower,
            upper,
            inclusive="both"
        )
        .sum()
    )

    actual_missing = int(
        values.isna()
        .sum()
    )

    print(
        f"\n--- CDC RANGE CHECK: {column} ---"
    )

    print(
        f"Range {lower}-{upper}:",
        f"{range_count:,}"
    )

    if range_count != expected_range_count:

        raise ValueError(
            f"{column} range count differs from CDC.\n"
            f"Expected: {expected_range_count:,}\n"
            f"Actual:   {range_count:,}"
        )

    for code, expected_count in special_counts.items():

        actual_count = int(
            values.eq(code)
            .sum()
        )

        print(
            f"Code {code}:",
            f"{actual_count:,}"
        )

        if actual_count != expected_count:

            raise ValueError(
                f"{column} code {code} count differs.\n"
                f"Expected: {expected_count:,}\n"
                f"Actual:   {actual_count:,}"
            )

    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )

    print(
        f"PASS: {column} CDC range summary matches."
    )


# ============================================================
# 16. DAYS / TIME VARIABLE CDC CHECKS
# ============================================================

#
# Vigorous work
#

check_range_summary(
    "PAQ610",
    1,
    7,
    1388,
    {
        77: 0,
        99: 1
    },
    4467
)


check_range_summary(
    "PAD615",
    10,
    840,
    1375,
    {
        7777: 0,
        9999: 6
    },
    4475
)


#
# Moderate work
#

check_range_summary(
    "PAQ625",
    1,
    7,
    2433,
    {
        77: 0,
        99: 6
    },
    3417
)


check_range_summary(
    "PAD630",
    10,
    840,
    2413,
    {
        7777: 0,
        9999: 13
    },
    3430
)


#
# Walking / bicycle transportation
#

check_range_summary(
    "PAQ640",
    1,
    7,
    1435,
    {
        77: 0,
        99: 4
    },
    4417
)


check_range_summary(
    "PAD645",
    10,
    660,
    1426,
    {
        7777: 0,
        9999: 4
    },
    4426
)


#
# Vigorous recreation
#

check_range_summary(
    "PAQ655",
    1,
    7,
    1433,
    {
        77: 0,
        99: 1
    },
    4422
)


check_range_summary(
    "PAD660",
    10,
    480,
    1430,
    {
        7777: 0,
        9999: 1
    },
    4425
)


#
# Moderate recreation
#

check_range_summary(
    "PAQ670",
    1,
    7,
    2305,
    {
        77: 0,
        99: 3
    },
    3548
)


check_range_summary(
    "PAD675",
    10,
    540,
    2300,
    {
        7777: 0,
        9999: 1
    },
    3555
)


#
# Sedentary activity
#
# CDC publishes:
#
#     0-1320 = 5,811
#     9999   = 35
#     Missing = 10
#

check_range_summary(
    "PAD680",
    0,
    1320,
    5811,
    {
        7777: 0,
        9999: 35
    },
    10
)


# ============================================================
# 17. ROUTING CHECK HELPER
# ============================================================

#
# The Yes/No activity question determines whether the
# corresponding DAYS variable is asked.
#
# CDC separately notes that time variables may be set
# missing during data editing when total reported activity
# reaches an implausible level.
#
# Therefore:
#
#     Yes -> days
#
# can be checked exactly.
#
# Do NOT require every Yes response to have a nonmissing
# minutes field after CDC editing.
#

def check_yes_to_days(
    parent_column,
    child_column
):

    expected = (
        df[parent_column]
        .eq(1)
        .to_numpy(
            dtype=bool
        )
    )

    present = (
        df[child_column]
        .notna()
        .to_numpy(
            dtype=bool
        )
    )

    mismatch_count = int(
        np.sum(
            expected != present
        )
    )

    print(
        f"\n--- ROUTING: "
        f"{parent_column}=1 -> {child_column} ---"
    )

    print(
        "Expected populated:",
        int(expected.sum())
    )

    print(
        "Actually populated:",
        int(present.sum())
    )

    print(
        "Routing mismatches:",
        mismatch_count
    )

    if mismatch_count != 0:

        raise ValueError(
            f"{child_column} routing does not "
            f"match {parent_column}=1."
        )

    print(
        "PASS: Routing validated."
    )


# ============================================================
# 18. ACTIVITY ROUTING CHECKS
# ============================================================

check_yes_to_days(
    "PAQ605",
    "PAQ610"
)


check_yes_to_days(
    "PAQ620",
    "PAQ625"
)


check_yes_to_days(
    "PAQ635",
    "PAQ640"
)


check_yes_to_days(
    "PAQ650",
    "PAQ655"
)


check_yes_to_days(
    "PAQ665",
    "PAQ670"
)


# ============================================================
# 19. ALLOWED CODE CHECKS
# ============================================================

ALLOWED_CATEGORY_VALUES = {

    "PAQ605": {
        1, 2, 7, 9
    },

    "PAQ620": {
        1, 2, 7, 9
    },

    "PAQ635": {
        1, 2, 7, 9
    },

    "PAQ650": {
        1, 2, 7, 9
    },

    "PAQ665": {
        1, 2, 7, 9
    }
}


print("\n--- ALLOWED CODE CHECK ---")


for col, allowed in ALLOWED_CATEGORY_VALUES.items():

    observed = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    unexpected = (
        observed - allowed
    )

    if unexpected:

        raise ValueError(
            f"{col} contains unexpected codes: "
            f"{sorted(unexpected)}"
        )

    print(
        f"PASS: {col} contains only "
        "CDC-documented codes."
    )


# ============================================================
# 20. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# The only numeric zeros expected after correction:
#
#     PAD680 = 3
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )

    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")

print(
    final_zero_counts
)


EXPECTED_FINAL_ZERO_COUNTS = {
    "PAD680": 3
}


if (
    final_zero_counts
    !=
    EXPECTED_FINAL_ZERO_COUNTS
):

    raise ValueError(
        "Final zero pattern differs from "
        "validated PAQ_J attachment.\n\n"
        f"Expected:\n{EXPECTED_FINAL_ZERO_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Final zero pattern contains exactly "
    "3 PAD680 zeros."
)


# ============================================================
# 21. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Only permitted numeric modification:
#
#     PAD680:
#
#     3 tiny XPORT zero artifacts -> 0
#
# Nothing else may change.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )

    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 17 variables match exact "
    "expected corrected XPT values."
)


# ============================================================
# 22. MISSING-VALUE FIDELITY BEFORE EXPORT
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    original_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    current_missing = (
        df[col]
        .isna()
        .to_numpy()
    )

    missing_mismatches += int(
        np.sum(
            original_missing
            !=
            current_missing
        )
    )


print(
    "\nMissing-position mismatches before export:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 23. EXPORT CSV
# ============================================================

#
# IMPORTANT FIDELITY POLICY
# ------------------------------------------------------------
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
# NO:
#
#     blanket decimal formatting
#
# PAQ_J itself contains only proven whole-number fields.
#
# However, this same export rule protects genuine
# Dietary/Laboratory/Examination decimal values in
# the broader NHANES workflow.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 24. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 25. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )

    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 17 CSV variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 26. CSV MISSING-VALUE FIDELITY
# ============================================================

csv_missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- CSV MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    csv_missing_mismatches
)


if csv_missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed "
        "during CSV export."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 27. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


if raw_csv_df.shape != (
    5856,
    17
):

    raise ValueError(
        "Raw CSV dimensions changed."
    )


# ============================================================
# 28. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 29. RAW CSV .0 CHECK
# ============================================================

#
# All PAQ_J fields were proven integral BEFORE
# Int64 conversion.
#
# Therefore it is safe to check these 17 fields
# for unnecessary trailing ".0".
#
# IMPORTANT:
#
# This check must NOT be copied indiscriminately
# to a mixed file containing genuine decimals.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]

    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )

    count = int(
        bad_mask.sum()
    )

    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' "
    "in any PAQ_J field."
)


# ============================================================
# 30. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    zero_count = int(
        values.eq(0)
        .sum()
    )

    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")

print(
    csv_zero_counts
)


if (
    csv_zero_counts
    !=
    EXPECTED_FINAL_ZERO_COUNTS
):

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_FINAL_ZERO_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "3 PAD680 zeros."
)


# ============================================================
# 31. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]

    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 32. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93705:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains missing SEQN."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN 93705-102956 valid "
    "and contains no '.0'."
)


# ============================================================
# 33. FINAL COMPLETE SOURCE -> CSV FIDELITY
# ============================================================

#
# Final redundant all-column fidelity check.
#

final_fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    final_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    ).to_numpy(
        dtype=float
    )

    same = np.allclose(
        expected_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        final_fidelity_failures.append(
            col
        )


if final_fidelity_failures:

    raise ValueError(
        "Final source -> CSV fidelity failures:\n"
        + str(final_fidelity_failures)
    )


print(
    "PASS: Final CSV preserves exact corrected "
    "numeric values for all 17 variables."
)


# ============================================================
# 34. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "PAQ_J CONVERSION COMPLETE"
)

print(
    "=" * 90
)

print(
    "Cycle: 2017-2018"
)

print(
    "Component: Questionnaire - Physical Activity"
)

print(
    "Eligible population: Age 18+"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny-value artifacts:",
    tiny_total
)

print(
    "PAD680 zero restorations:",
    correction_count
)

print(
    "Original ordinary numeric zeros:",
    true_zero_total
)

print(
    "Genuine decimal variables: NONE"
)

print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)

print(
    "PAQ605 vigorous work = Yes:",
    int(
        df["PAQ605"]
        .eq(1)
        .sum()
    )
)

print(
    "PAQ620 moderate work = Yes:",
    int(
        df["PAQ620"]
        .eq(1)
        .sum()
    )
)

print(
    "PAQ635 walk/bicycle = Yes:",
    int(
        df["PAQ635"]
        .eq(1)
        .sum()
    )
)

print(
    "PAQ650 vigorous recreation = Yes:",
    int(
        df["PAQ650"]
        .eq(1)
        .sum()
    )
)

print(
    "PAQ665 moderate recreation = Yes:",
    int(
        df["PAQ665"]
        .eq(1)
        .sum()
    )
)

print(
    "PAD680 sedentary minutes = 0:",
    int(
        df["PAD680"]
        .eq(0)
        .sum()
    )
)

print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)

print(
    "Missing-position differences: "
    "0 expected"
)

print(
    "Tiny values remaining: "
    "0 expected"
)

print(
    "Unnecessary '.0': "
    "0 expected"
)

print(
    "Scientific notation: "
    "0 expected"
)

print(
    "=" * 90
)

NHANES 2017-2018 SMQ_J XPT -> CSV
Rows: 6,724
Columns: 37
PASS: Structure = 6,724 rows x 37 columns.

--- FIELD TYPE PLAN ---
Character fields: ['SMDUPCA', 'SMD100BR']
Genuine decimal fields: ['SMD100NI']
Whole-number/code numeric fields: 34


ValueError: SMDUPCA unexpectedly did not load as a character/object field.